# Deploy Intelligent RAG Agent to AgentCore Runtime

This notebook demonstrates how to deploy the Strands-based intelligent RAG agent to Amazon Bedrock AgentCore Runtime.

### Why Deploy to AgentCore Runtime?

Previously, our intelligent RAG agent ran locally in the development environment. While this works for prototyping, production applications need:

- **Internet Accessibility**: Make your agent available as a service that can be invoked from anywhere
- **Automatic Scaling**: Handle varying loads without manual infrastructure management
- **Enterprise Observability**: Built-in CloudWatch monitoring, traces, and metrics
- **Memory Capabilities**: Maintain conversation context and learn user preferences over time
- **Serverless Infrastructure**: No servers to manage, pay only for what you use

AgentCore Runtime transforms your local agent prototype into a production-ready service by handling all infrastructure complexity, security, and operational concerns.

In this lab, we have a Strands Agent in seperate python files.  
You can invoke the agents by executing `intelligent_rag_agent_runtime.py` locally.

They have 2 tools to help them lookup data.
* Structured data is handled by `structured_data_assistant()`
* Unstructured data is handled by `unstructured_data_assistant()`.

This lab walks you through deploying the agents to AgentCore runtime while leveraging AgentCore Memory and Observability capabilities.

![agent_core](../images/agentcore.png)



## Prerequisites
- Complete previous labs to set up the knowledge bases
- Knowledge base IDs stored in variables
- AgentCore runtime permissions configured

Install the AgentCore SDK and starter toolkit to enable deployment and management of agents on the AgentCore platform:

In [1]:
# Install required packages
!uv pip install bedrock-agentcore bedrock-agentcore-starter-toolkit --quiet

Import required libraries and initialize the AgentCore Runtime client to interact with the deployment service:

In [2]:
import json
import os
import random, string
import time

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = boto3.Session()
region = boto_session.region_name

print(f"current region: {region}")
account_id = boto_session.client("sts").get_caller_identity()["Account"]
print(f"current account: {account_id}")

agentcore_runtime = Runtime()

current region: us-west-2
current account: 298894475115


Get the current notebook's directory path to locate the agent code and policy files for deployment:

In [3]:
import os
from pathlib import Path

import ipynbname

try:
    # Get the notebook name and path
    notebook_path = ipynbname.path()
    notebook_dir = Path(notebook_path.parent)
except:
    notebook_dir = Path.resolve()

print(f"notebook_dir: {notebook_dir}")

# Agentcore starter toolkit expects the files in the current directory
# So changing our working directory.
os.chdir(notebook_dir)
print(f"changed working directory to: {notebook_dir}")

notebook_dir: /Workshop/Lab 3 - AgentCore Deployment and Memory
changed working directory to: /Workshop/Lab 3 - AgentCore Deployment and Memory


### Create IAM Execution Role

The AgentCore runtime requires an IAM role with specific permissions to:
- Access Bedrock models for AI inference
- Retrieve from knowledge bases (KB)
- Write to CloudWatch logs
- Access SSM Parameter for KB ID
- Pull/push Docker images to ECR

We'll create this role using our predefined policy templates that include all necessary permissions.

In [4]:
# Load policy from external file
with open('policy.json', 'r') as f:
    policy_template = f.read()

# load trust policy from external file
with open('trust-policy.json', 'r') as f:
    trust_policy_template = f.read()

policy = policy_template.replace('REGION', region).replace('ACCOUNT_ID', account_id).replace('REPO_ARN', '*')
print(f"Policy loaded and updated with region: {region}, account: {account_id}")

trust_policy = trust_policy_template.replace('REGION', region).replace('ACCOUNT_ID', account_id)

# create IAM role using the policies
suffix = random.choices(string.ascii_lowercase + string.digits, k=8)
iam_client = boto3.client('iam')
role_name = f"bedrock-runtime-execution-role-{''.join(suffix)}"

role = iam_client.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=trust_policy
)

iam_client.put_role_policy(
    RoleName=role_name,
    PolicyName='bedrock-runtime-execution-policy',
    PolicyDocument=policy
)

Policy loaded and updated with region: us-west-2, account: 298894475115


{'ResponseMetadata': {'RequestId': 'c0dfb3f6-4968-4aa3-985f-345988aaf8a2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 12 Mar 2026 15:34:45 GMT',
   'x-amzn-requestid': 'c0dfb3f6-4968-4aa3-985f-345988aaf8a2',
   'content-type': 'text/xml',
   'content-length': '206'},
  'RetryAttempts': 0}}

### Agent Configuration Process

The `configure()` method prepares your agent for deployment by:

1. **Creating Infrastructure**: ECR repository for Docker images, IAM roles if needed
2. **Generating Dockerfile**: Configures Python environment, installs dependencies, sets up OpenTelemetry
3. **Creating .dockerignore**: Excludes unnecessary files from the container
4. **Generating .bedrock_agentcore.yaml**: Configuration file that defines:
   - Entrypoint script (your agent code)
   - Execution role ARN
   - Region and agent name
   - Runtime requirements

This configuration is saved locally and used during the launch process.

**Agent Code Location:**

The agent implementation is in `intelligent_rag_agent_runtime.py` in this directory. Take a moment to review the code to understand:
- How the agent routes queries between structured and unstructured knowledge bases
- The tool definitions for querying each knowledge base type
- The agent's decision-making logic for selecting the appropriate data source

**What happens during configuration:**
- Creates ECR repository and IAM role (if needed)
- Generates Dockerfile and .dockerignore for containerization
- Creates `.bedrock_agentcore.yaml` configuration file
- Packages your agent code (`intelligent_rag_agent_runtime.py`) for deployment

In [5]:
agent_name = "intelligent_rag_agent"
response = agentcore_runtime.configure(
    entrypoint="intelligent_rag_agent_runtime.py",
    execution_role=role['Role']['Arn'],
    # auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

Entrypoint parsed: file=/Workshop/Lab 3 - AgentCore Deployment and Memory/intelligent_rag_agent_runtime.py, bedrock_agentcore_name=intelligent_rag_agent_runtime
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: intelligent_rag_agent
Memory disabled
Network mode: PUBLIC
Generated .dockerignore
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Workshop/Lab 3 - AgentCore Deployment and Memory/.dockerignore
Setting 'intelligent_rag_agent' as default agent
Bedrock AgentCore configured: /Workshop/Lab 3 - AgentCore Deployment and Memory/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Workshop/Lab 3 - AgentCore Deployment and Memory/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Workshop/Lab 3 - AgentCore Deployment and Memory/Dockerfile'), dockerignore_path=PosixPath('/Workshop/Lab 3 - AgentCore Deployment and Memory/.dockerignore'), runtime='Docker', runtime_type=None, region='us-west-2', account_id='298894475115', execution_role='arn:aws:iam::298894475115:role/bedrock-runtime-execution-role-k283l94o', ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

### CodeBuild Deployment Process

When you call `launch()`, AgentCore triggers a CodeBuild job that:

**Pre-build Phase:**
- Authenticates with ECR registry
- Prepares build environment

**Build Phase:**
- Creates ARM64 Docker image (optimized for AWS Graviton processors)
- Installs Python dependencies from requirements.txt
- Copies your agent code into the container
- Configures OpenTelemetry instrumentation

**Post-build Phase:**
- Pushes Docker image to ECR
- Creates AgentCore runtime endpoint
- Configures auto-scaling and monitoring

The entire process takes 3-5 minutes. You can monitor progress through the status checks.

In [6]:
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'intelligent_rag_agent' to account 298894475115 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: intelligent_rag_agent
ECR repository available: 298894475115.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-intelligent_rag_agent
Using execution role from config: arn:aws:iam::298894475115:role/bedrock-runtime-execution-role-k283l94o
Preparing CodeBuild project and uploading source...


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-intelligent_rag_agent


Getting or creating CodeBuild execution role for agent: intelligent_rag_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-1d0cbcd5fa
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-1d0cbcd5fa
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-1d0cbcd5fa
✓ Role created: arn:aws:iam::298894475115:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-1d0cbcd5fa
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-1d0cbcd5fa
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::298894475115:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-1d0cbcd5fa
Created S3 bucket: bedrock-agentcore-codebuild-sources-298894475115-us-west-2
Using dockerignore.template with 43 patterns for zip filtering
Uploaded source to S3: intelligent_rag_agent/source.zip
Created CodeBuild project: bedrock-agentcore-inte

Monitor the deployment status to ensure the agent is ready before attempting to invoke it:

In [7]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
print(status)

# Store the agent ARN for use in other notebooks
if status == 'READY':
    try:
        endpoint = status_response.endpoint
        # Get the agent ARN from the status response
        if 'agentRuntimeArn' in endpoint:
            agent_arn = endpoint['agentRuntimeArn']
        elif 'endpointName' in endpoint:
            # Construct ARN from endpoint name if not directly available
            endpoint_name = endpoint['endpointName']
            agent_arn = f"arn:aws:bedrock:{region}:{account_id}:agent-runtime/{endpoint_name}"
        else:
            raise ValueError("Could not extract agent ARN from status response")

        %store agent_arn
        print(f"Agent ARN stored for use in other notebooks: {agent_arn}")
    except Exception as e:
        print(f"Could not store agent ARN: {e}")

Retrieved Bedrock AgentCore status for: intelligent_rag_agent


READY
Stored 'agent_arn' (str)
Agent ARN stored for use in other notebooks: arn:aws:bedrock-agentcore:us-west-2:298894475115:runtime/intelligent_rag_agent-wbztS3FhxY


### Testing Your Deployed Agent

Now that the agent is deployed to AgentCore Runtime, you can invoke it just like you did locally, but with key differences:

- **Invocation Method**: Uses `agentcore_runtime.invoke()` instead of direct Python execution
- **Scalability**: Automatically scales based on request volume
- **Observability**: Every invocation generates traces and metrics in CloudWatch
- **Availability**: Accessible via API endpoint, not just local execution

The agent maintains the same intelligent routing behavior between structured and unstructured knowledge bases, but now runs in a production-ready environment.

**Helper Function for Response Formatting**

The `print_response_text()` function extracts and displays the text content from API responses in a clean, readable format:

In [8]:
import json


def print_response_text(invoke_response):
    response = invoke_response['response']
    
    # If it's a list, join all parts first
    if isinstance(response, list):
        response = ''.join(response)
    
    # Parse JSON
    response = json.loads(response)
    
    # Extract the text content
    text = response['result']['content'][0]['text']
    print(text)

**Convenience Function for Agent Invocation**

Define a convenience function to simplify agent invocations and format responses consistently:

In [9]:
def ask(question):
    print(f"Query: {question}")
    print("-" * 100)
    response = agentcore_runtime.invoke({"prompt": question})
    print_response_text(response)
    print("\n")

#### Queries

Test the deployed agent with sample queries to verify it can route between structured and unstructured knowledge bases:

In [10]:
ask("How many customers reviewed product_890, are those reviews positive or negative?")

Query: How many customers reviewed product_890, are those reviews positive or negative?
----------------------------------------------------------------------------------------------------
Based on the data retrieved, here's a summary of reviews for **product_890**:

## Review Count
**4 customers** have reviewed product_890.

## Sentiment Analysis
The reviews are **predominantly negative**:

| Rating | Count | Sentiment |
|--------|-------|-----------|
| 1 star | 2 | Highly Negative |
| 2 stars | 1 | Negative |
| 3 stars | 1 | Neutral/Mixed |

### Key Issues Identified:
1. **Quality Problems**: Multiple reviewers report that the mixing bowl set (kitchenware) has structural issues - warping with hot ingredients, cracking, and handles breaking
2. **Poor Durability**: Customers report the product failing after minimal use (one use, two uses)
3. **Material Concerns**: The cheap plastic and flimsy construction are frequently mentioned
4. **Value for Money**: Reviewers feel the product is ov

In [11]:
ask("What are customer complaints about the lowest rated product?")

Query: What are customer complaints about the lowest rated product?
----------------------------------------------------------------------------------------------------
## Customer Complaints for the Lowest Rated Product: **product_890**

**Product Type:** Mixing Bowl Set (Kitchenware)  
**Average Rating:** 1 out of 5 stars

### Major Complaint Categories:

#### 1. **Durability & Structural Failure** ⚠️
- **Handles breaking**: Broke off after just 2 uses
- **Bowls cracking**: Complete failure with cracks running down the middle
- **Warping**: Plastic warps when exposed to hot ingredients
- **Impact**: Products became completely unusable after minimal use

#### 2. **Poor Material Quality** 
- Cheap plastic construction
- Flimsy handles and components
- Material feels fragile ("would crack if you looked at it wrong")
- Feels like poor quality compared to dollar store products

#### 3. **Design Flaws**
- Weak bowl structure - smallest bowl feels flimsy
- Non-slip grip on bottom doesn't wo

In [12]:
ask("What products have the best reviews?")

Query: What products have the best reviews?
----------------------------------------------------------------------------------------------------
Based on the comprehensive data retrieved, here's a summary of the best-reviewed products:

## Top-Rated Products (4.0-Star Average Rating)

There are **28 products with a 4-star average rating**, which represents the highest tier of customer satisfaction. Here are the best-reviewed products:

### 4-Star Rated Products:
**product_763, product_468, product_280, product_184, product_562, product_561, product_228, product_662, product_59, product_848, product_856, product_63, product_394, product_374, product_676, product_315, product_244, product_490, product_83, product_851, product_899, product_743, product_172, product_691, product_611, product_321**

## Product Rating Distribution Summary:

| Average Rating | Number of Products |
|---|---|
| **4 stars** | 26 |
| **3 stars** | 568 |
| **2 stars** | 392 |
| **1 star** | 11 |

## Key Insights:


## Try making up your own queries!
Now it's your turn! Try creating some queries to test the system:

**Examples you could try:**
- 'What are the main features of product X?'
- 'How does service Y compare to competitors?'
- 'What are the pricing options for Z?'
- 'Can you explain the benefits of using this solution?'
Feel free to experiment with different types of questions!


In [13]:
# Try your own queries here using the ask() function
# Example: ask("Your question here")
ask("What is the product with the highest purchase spend?")

Query: What is the product with the highest purchase spend?
----------------------------------------------------------------------------------------------------
## Product with Highest Purchase Spend

**Product ID:** `6dca6270-8a95-4d0b-91ab-128fb500ca9a`

**Total Purchase Spend:** $12,134.01

### Top 5 Products by Purchase Spend:

| Rank | Product ID | Total Spend |
|------|---------|------------|
| 1 | 6dca6270-8a95-4d0b-91ab-128fb500ca9a | **$12,134.01** |
| 2 | 51d01161-e41c-4aa3-bbaf-61d0af13f5ae | $11,451.12 |
| 3 | 31330b26-25bd-4204-9d82-fba258706566 | $11,345.43 |
| 4 | 4b9b432f-b524-44f2-99ad-da192022a080 | $10,974.84 |
| 5 | 2a763351-29e4-444f-9486-24efee8679fb | $10,876.05 |

This product with ID `6dca6270-8a95-4d0b-91ab-128fb500ca9a` is your top revenue generator with over $12,100 in total purchase spend, making it your most commercially successful product.




## Next Steps

Your agent is now deployed to AgentCore Runtime and ready for production use!

**Ready to continue?** Proceed to [**Lab 3.2**](3.2-agentcore-memory.ipynb) to add AgentCore Memory capabilities for conversation context and user preferences.
